# Figure 1: sampling locations and population structure of *An. dirus* s.l.

(a) Sampling map and (b) PCA of 100,000 SNPs from the longest contig (KB672490), using the Adir1.0 data via the `malariagen_data` API. The PCA defines the analysis cohorts (`ADIR1_COHORTS`) used in Figures 2 and 3.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import malariagen_data
import plotly.express as px

## Setup

In [ ]:
# Results cache shared by all notebooks, so expensive analyses (PCA, ROH, GWSS)
# are only computed once
RESULTS_CACHE = Path("~/sea_vector_cache").expanduser()
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)


# In malariagen_data 15.10.0, Adir1 `variant_allele` arrays are object dtype, which
# zarr cannot write to the results cache (ValueError: missing object_codec for
# object array). Cast them to bytes before caching.
class Adir1(malariagen_data.Adir1):
    def results_cache_set(self, *, name, params, results):
        results = {k: v.astype("S") if v.dtype == object else v for k, v in results.items()}
        super().results_cache_set(name=name, params=params, results=results)

In [ ]:
# An. dirus s.l. cohorts, defined from the PCA in Figure 1B. Four samples from
# Cambodian site RS23 cluster with An. baimaii.
RS23_BAIMAII = ["S1214", "S1312", "S1047", "S1371"]

ADIR1_COHORTS = {
    "pca_groupa": f"country == 'Thailand' or partner_sample_id in {RS23_BAIMAII}",
    "pca_groupb": f"country == 'Cambodia' and partner_sample_id not in {RS23_BAIMAII}",
    "pca_groupc": "country == 'Bangladesh'",
}

ADIR1_COHORT_LABELS = {
    "pca_groupa": "baimaii-Cambodia/Thailand",
    "pca_groupb": "dirus_ss-Cambodia",
    "pca_groupc": "baimaii-Bangladesh",
}

COUNTRY_COLOURS = {
    "Bangladesh": "#e41a1c",
    "Cambodia": "#377eb8",
    "Thailand": "#4daf4a",
}

In [ ]:
adir1 = Adir1(results_cache=str(RESULTS_CACHE))
df_samples = adir1.sample_metadata()
df_samples.groupby(["country", "taxon"]).size()

## Figure 1A: sampling locations

In [ ]:
def plot_sample_map(df, bbox, colour_by="country", colours=COUNTRY_COLOURS):
    fig = plt.figure(figsize=(6, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(bbox)

    ax.add_feature(cfeature.LAND, zorder=0, edgecolor="black", facecolor="#f0f0f0")
    ax.add_feature(cfeature.OCEAN, zorder=0, facecolor="#c6dbef")
    ax.add_feature(cfeature.BORDERS, linestyle="--", edgecolor="gray")
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.RIVERS, edgecolor="#3182bd")

    country_label_coords = {
        "Thailand": (100.5, 15.0),
        "Bangladesh": (92.0, 23.2),
        "Cambodia": (105.0, 14.5),
    }
    for country, (lon, lat) in country_label_coords.items():
        ax.text(
            lon, lat, country,
            transform=ccrs.PlateCarree(),
            fontsize=14, fontweight="bold", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7),
        )

    # One marker per unique collection location
    df_locs = df.drop_duplicates(["latitude", "longitude", colour_by])
    for value, d in df_locs.groupby(colour_by):
        ax.scatter(
            d["longitude"], d["latitude"],
            color=colours[value], s=60, edgecolor="black",
            transform=ccrs.PlateCarree(), label=value,
        )
    ax.legend(title="Country")

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.3, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    plt.tight_layout()
    return fig

In [ ]:
fig = plot_sample_map(df_samples, bbox=[90, 110, 5, 24])
fig.savefig(FIGURES_DIR / "fig1a_map.svg", bbox_inches="tight", dpi=300)

## Figure 1B: PCA

In [ ]:
df_pca, evr = adir1.pca(region="KB672490", n_snps=100_000, n_components=10)

In [ ]:
fig = px.scatter(
    df_pca,
    x="PC1",
    y="PC2",
    color="country",
    color_discrete_map=COUNTRY_COLOURS,
    hover_data=["partner_sample_id", "location", "country", "year", "taxon"],
    width=500,
    height=400,
    template="simple_white",
)
fig.update_layout(
    font=dict(size=14),
    xaxis_title=f"PC1, {evr[0] * 100:.2f}%",
    yaxis_title=f"PC2, {evr[1] * 100:.2f}%",
    legend_title="Country",
)
fig.write_image(FIGURES_DIR / "fig1b_pca.svg")
fig.show()

## Analysis cohorts

PC1 separates *An. baimaii* (PC1 > 0) from *An. dirus* s.s. (PC1 < 0). Note that the `taxon` column in the Adir1.0 metadata labels the Thai samples as *dirus*; the cohorts below follow the PCA instead. Four samples from Cambodian site RS23 cluster with *An. baimaii*.

Check that the cohort queries agree with the PCA:

In [ ]:
for cohort, query in ADIR1_COHORTS.items():
    df_pca.loc[df_pca.eval(query), "cohort"] = ADIR1_COHORT_LABELS[cohort]

df_pca.groupby(["cohort", "country"]).agg(
    n=("PC1", "size"), PC1_min=("PC1", "min"), PC1_max=("PC1", "max")
)

In [ ]:
# An. dirus s.s. and An. baimaii should fall on opposite sides of PC1 = 0
is_dirus = df_pca["cohort"] == ADIR1_COHORT_LABELS["pca_groupb"]
pc1_sign = df_pca["PC1"] > 0
assert pc1_sign[is_dirus].nunique() == 1
assert pc1_sign[~is_dirus].nunique() == 1
assert pc1_sign[is_dirus].iloc[0] != pc1_sign[~is_dirus].iloc[0]